<a href="https://colab.research.google.com/github/pratik111111/Medical_Model_Q-A/blob/main/M_Q%26A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import json
import torch
import numpy as np
import re
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import accuracy_score
from collections import Counter
from dataclasses import dataclass
from transformers.tokenization_utils_base import PreTrainedTokenizerBase
from typing import Optional, Union
import torch

# ==============================
# 3. LOAD DATA
# ==============================
with open("medical_meadow_medqa.json", "r", encoding="utf-8") as f:
    raw_data = json.load(f)

print("Total raw samples:", len(raw_data))

Total raw samples: 10178


In [ ]:
# ==============================
# 4. PARSE OPTIONS FROM TEXT
# ==============================
label_map = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
processed_data = []

for item in raw_data:
    try:
        full_input = item["input"]
        answer = item["output"]
        label_letter = answer.split(":")[0].strip()

        if label_letter not in label_map:
            continue

        # Extract question (before the options dict)
        question_match = re.split(r"\n\s*\{", full_input)
        question = question_match[0].strip()

        # Extract options dict from input
        options_match = re.search(r"\{([^}]+)\}", full_input)
        if not options_match:
            continue

        options_str = "{" + options_match.group(1) + "}"
        # safely parse: 'A': 'text', ...
        options = {}
        for match in re.finditer(r"'([ABCDE])':\s*'([^']+)'", options_str):
            options[match.group(1)] = match.group(2)

        if len(options) < 2:
            continue

        processed_data.append({
            "question": question,
            "option_a": options.get("A", ""),
            "option_b": options.get("B", ""),
            "option_c": options.get("C", ""),
            "option_d": options.get("D", ""),
            "option_e": options.get("E", ""),
            "label": int(label_map[label_letter])
        })

    except Exception as e:
        continue

print("Processed samples:", len(processed_data))
print("Label distribution:", Counter(d["label"] for d in processed_data))
print("\nSample item:")
print("Q:", processed_data[0]["question"][:100])
print("A:", processed_data[0]["option_a"])
print("Label:", processed_data[0]["label"])

In [18]:
# ==============================
# 5. CREATE DATASET
# ==============================
dataset = Dataset.from_list(processed_data)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

In [ ]:
# ==============================
# 6. LOAD TOKENIZER & MODEL
# ==============================
model_name = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# ==============================
# 7. TOKENIZE FOR MULTIPLE CHOICE
# ==============================
option_keys = ["option_a", "option_b", "option_c", "option_d", "option_e"]

def tokenize_mc(batch):
    questions = batch["question"]

    # For each of 5 options, tokenize [question, option] pair
    all_encodings = [
        tokenizer(
            questions,
            batch[opt],
            padding="max_length",
            truncation=True,
            max_length=256
        )
        for opt in option_keys
    ]

    # Stack: shape = [batch, 5, seq_len]
    return {
        "input_ids": [
            [all_encodings[i]["input_ids"][j] for i in range(5)]
            for j in range(len(questions))
        ],
        "attention_mask": [
            [all_encodings[i]["attention_mask"][j] for i in range(5)]
            for j in range(len(questions))
        ],
        "label": batch["label"]
    }

tokenized = dataset.map(tokenize_mc, batched=True,
                        remove_columns=dataset["train"].column_names)
tokenized.set_format(type="torch")

In [ ]:
# ==============================
# 8. LOAD MODEL
# ==============================
model = AutoModelForMultipleChoice.from_pretrained(
    model_name,
    num_labels=5
)

# ==============================
# 9. TRAINING CONFIG
# ==============================
training_args = TrainingArguments(
    output_dir="./results_mc",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,   # MC uses 5x memory
    per_device_eval_batch_size=4,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=torch.cuda.is_available()
)

In [21]:
# ==============================
# 10. METRICS
# ==============================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    return {"accuracy": accuracy_score(labels, preds)}

# ==============================
# 11. TRAINER
# ==============================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
# ==============================
# 12. TRAIN
# ==============================
trainer.train()

In [ ]:
# ==============================
# 13. EVALUATE
# ==============================
results = trainer.evaluate()
print("Evaluation Results:", results)

In [ ]:
# ==============================
# 14. SAVE
# ==============================
model.save_pretrained("./medical_model_mc")
tokenizer.save_pretrained("./medical_model_mc")
print("Model saved!")

In [ ]:
# ==============================
# 15. PREDICT
# ==============================
reverse_map = {v: k for k, v in label_map.items()}

def predict(item):
    device = next(model.parameters()).device
    question = item["question"]
    options = [item[k] for k in option_keys]

    encodings = tokenizer(
        [question] * 5,
        options,
        padding="max_length",
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    input_ids = encodings["input_ids"].unsqueeze(0).to(device)
    attention_mask = encodings["attention_mask"].unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

    pred = torch.argmax(outputs.logits, dim=1).item()
    return reverse_map[pred]
    # Test on first sample
sample = processed_data[5]
print("Prediction:", predict(sample))
print("Correct:", raw_data[5]["output"])


In [35]:
import IPython
IPython.display.clear_output()